# IMP38

Done:
- Werfyfikacja czy repo lokalnie działa?
- Werfyikacja czy podczas treningu nie napotkano błędów
- PDF z napotkanymi błędami oraz jak je rozwiązano 
- Dane LUNA przeniesione z .146 na .76
- Setup treningu przygotowany 
To do:
- Wymagany Python 3.8 -> akceptacja ToS Anacondy?
- Wymagany kontakt z administratorem d.t komendy unzip/bdstar, unzipowanie przez python3 też nie działa
- Uruchomienie treningu na 10 foldów 
----------------------------------------------------------------------------------------------------------
- Sanity check czy model nie overfittuje / inne problemy + metryki* (nnDetection wspiera natywnie MLFlow)
- *MaP, IoU, loss (w nnDetection jest składany)
- Dalszy kontakt z Magą co należy zrobić w sprawie modelu
- Testy manualne IMP38 


# Synthetic dataset

### set HF token

In [ ]:
#token
import os
os.environ["HF_TOKEN"] = ""

## other sources of data mimicing medical notes:

- https://www.physionet.org/content/mimic-iv-note/2.2/
- https://huggingface.co/datasets/zhengyun21/PMC-Patients

## download models to ./models dir
Run:
- hf download Qwen/Qwen2.5-7B-Instruct --local-dir ./Qwen2.5-7B-Instruct
- hf download m42-health/Llama3-Med42-8B --local-dir ./Llama3-Med42-8B
- hf download microsoft/MediPhi-Instruct --local-dir ./MediPhi-Instruct
- hf download google/medgemma-27b-text-it --local-dir ./medgemma-27b-text-it

1. https://huggingface.co/Qwen/Qwen2.5-7B-Instruct

2. https://huggingface.co/m42-health/Llama3-Med42-8B

3. https://huggingface.co/microsoft/MediPhi-Instruct

4. https://huggingface.co/google/medgemma-27b-text-it

*open-bio-llm


## initialize models from local dir

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os


model_id = "mistralai/Mistral-7B-Instruct-v0.3"
local_dir = "/home/test/caise-ner/models" 

if not os.path.exists(local_dir):
    print(f"Pobieranie modelu {model_id} do {local_dir}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    # use_safetensors=True gwarantuje ominięcie błędów wersji Torcha
    model = AutoModelForCausalLM.from_pretrained(
        model_id, 
        use_safetensors=True, 
        torch_dtype=torch.float16
    )
    
    tokenizer.save_pretrained(local_dir)
    model.save_pretrained(local_dir)
    print("Model pobrany pomyślnie!")


In [3]:

tokenizer = AutoTokenizer.from_pretrained(local_dir)
model = AutoModelForCausalLM.from_pretrained(
    local_dir,
    device_map="auto",
    torch_dtype=torch.float16,
    use_safetensors=True
)

# Przykładowy prompt do generowania dokumentacji medycznej
prompt = "Generate a synthetic medical discharge summary for a patient with acute appendicitis. Include: History of present illness, Physical exam, and Recommendations."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=400, temperature=0.7, do_sample=True)

print("\n--- WYGENEROWANA NOTATKA MEDYCZNA ---\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "/home/test/caise-ner/models/MediPhi-Instruct"
torch_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (torch.float16 if torch.cuda.is_available() else torch.float32)
device_map = "auto" if torch.cuda.is_available() else None

tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch_dtype,
    device_map=device_map,
    trust_remote_code=True,
)

print(f"Model loaded: {model_path}")